# Assignment 4 — Loan Approval Classification

**Dataset:** `loan_data_clean.csv`  
**Target:** `Loan_Status` (Y = Approved, N = Denied)  
**Models:** Logistic Regression & Random Forest

## 1. Load Data & Explore

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, ConfusionMatrixDisplay
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('loan_data_clean.csv')
print(df.shape)
df.head()

In [ ]:
df.dtypes

In [ ]:
df.describe()

In [ ]:
# Class balance
df['Loan_Status'].value_counts(normalize=True).plot(kind='bar', color=['steelblue','salmon'], edgecolor='black')
plt.title('Loan Status Distribution')
plt.xlabel('Loan Status (Y=Approved, N=Denied)')
plt.ylabel('Proportion')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
print(df['Loan_Status'].value_counts())

## 2. Feature Selection & Justification

The dataset contains 12 potential predictor columns. I selected **6 features** and excluded the rest based on the reasoning below.

### Selected Features

| Feature | Type | Justification |
|---|---|---|
| `Credit_History` | Binary (0/1) | Strongest single predictor of loan repayment — lenders rely on this above all else |
| `ApplicantIncome` | Continuous | Higher income directly supports repayment capacity |
| `LoanAmount` | Continuous | Larger loans are harder to repay; models the debt burden |
| `Education` | Categorical | Graduate status is a proxy for income stability and employment prospects |
| `Married` | Categorical | Married applicants often have dual income, reducing default risk |
| `Property_Area` | Categorical | Urban/Semiurban/Rural reflects local economic conditions and collateral value |

### Excluded Features

| Feature | Reason for Exclusion |
|---|---|
| `Loan_ID` | Unique row identifier — carries no predictive signal |
| `Gender` | Weak predictor in this dataset; excluding avoids encoding demographic bias into the model |
| `CoapplicantIncome` | Majority of values are 0 (single applicants), making it sparse and mostly redundant with `ApplicantIncome` |
| `Loan_Amount_Term` | Nearly all loans use a 360-month term (>95%), so variance is negligible and it adds noise |
| `Self_Employed` | Low variance and high missingness in similar datasets; income reliability already captured by `ApplicantIncome` |
| `Dependents` | Indirect financial signal already captured by `Married` and `ApplicantIncome`; adds noise without clear gain |

## 3. Preprocessing — Encoding & Train/Test Split

In [ ]:
features = ['Credit_History', 'ApplicantIncome', 'LoanAmount',
            'Education', 'Married', 'Property_Area']
target = 'Loan_Status'

data = df[features + [target]].copy()

# Encode binary categoricals
le = LabelEncoder()
for col in ['Education', 'Married', 'Property_Area', 'Loan_Status']:
    data[col] = le.fit_transform(data[col])

print("Encoding check:")
print("  Education  :", sorted(df['Education'].unique()), "->", sorted(data['Education'].unique()))
print("  Married    :", sorted(df['Married'].unique()), "->", sorted(data['Married'].unique()))
print("  Property   :", sorted(df['Property_Area'].unique()), "->", sorted(data['Property_Area'].unique()))
print("  Loan_Status:", sorted(df['Loan_Status'].unique()), "->", sorted(data['Loan_Status'].unique()))
data.head()

In [ ]:
X = data[features]
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale continuous features (critical for Logistic Regression)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train size: {len(X_train)}  |  Test size: {len(X_test)}")
print(f"Class distribution in test set:\n{y_test.value_counts()}")

## 4. Model 1 — Logistic Regression

### Algorithm Justification

Logistic Regression is a strong baseline for binary classification. Key reasons for choosing it here:

- **Interpretability:** Coefficients directly show each feature's directional effect on approval odds, which matters in a lending context.
- **Works well with mixed features:** Handles both continuous and encoded categorical inputs after scaling.
- **Performs well on linearly separable data:** Credit history and income are likely strong linear separators of approval/denial.
- **Low variance, low risk of overfitting** on a small dataset (480 rows) — a complex model like a deep neural net would likely overfit.
- **Weakness:** Assumes a roughly linear decision boundary; if the true boundary is highly non-linear, it will underfit.

In [ ]:
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_sc, y_train)

y_pred_lr   = lr.predict(X_test_sc)
y_proba_lr  = lr.predict_proba(X_test_sc)[:, 1]

acc_lr  = accuracy_score(y_test, y_pred_lr)
auc_lr  = roc_auc_score(y_test, y_proba_lr)

print(f"Logistic Regression — Accuracy: {acc_lr:.4f}  |  AUC: {auc_lr:.4f}")
print()
print(classification_report(y_test, y_pred_lr, target_names=['Denied (0)', 'Approved (1)']))

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay.from_estimator(lr, X_test_sc, y_test,
                                       display_labels=['Denied','Approved'],
                                       colorbar=False, ax=ax)
ax.set_title('Logistic Regression — Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Feature coefficients (log-odds)
coef_df = pd.DataFrame({'Feature': features, 'Coefficient': lr.coef_[0]})
coef_df = coef_df.reindex(coef_df['Coefficient'].abs().sort_values(ascending=True).index)

coef_df.plot(kind='barh', x='Feature', y='Coefficient', legend=False,
             color=['salmon' if c < 0 else 'steelblue' for c in coef_df['Coefficient']],
             edgecolor='black')
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Logistic Regression — Feature Coefficients (log-odds)')
plt.xlabel('Coefficient')
plt.tight_layout()
plt.show()

## 5. Model 2 — Random Forest

### Algorithm Justification

Random Forest is an ensemble of decision trees that reduces variance through bagging and random feature subsets. Key reasons for choosing it here:

- **Handles non-linear interactions:** Can capture joint effects (e.g., high income AND high loan amount) that Logistic Regression models linearly.
- **Robust to outliers:** Income data often has high outliers; tree-based splits are not affected by their magnitude.
- **Built-in feature importance:** Easily identifies which features drive predictions most.
- **Works without scaling:** No need to standardize features, since splits are rank-based.
- **Weakness:** More prone to overfitting on small datasets than Logistic Regression; less interpretable at the individual-tree level.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf.fit(X_train, y_train)   # unscaled — trees don't need scaling

y_pred_rf   = rf.predict(X_test)
y_proba_rf  = rf.predict_proba(X_test)[:, 1]

acc_rf  = accuracy_score(y_test, y_pred_rf)
auc_rf  = roc_auc_score(y_test, y_proba_rf)

print(f"Random Forest — Accuracy: {acc_rf:.4f}  |  AUC: {auc_rf:.4f}")
print()
print(classification_report(y_test, y_pred_rf, target_names=['Denied (0)', 'Approved (1)']))

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay.from_estimator(rf, X_test, y_test,
                                       display_labels=['Denied','Approved'],
                                       colorbar=False, ax=ax)
ax.set_title('Random Forest — Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importances
fi_df = pd.DataFrame({'Feature': features, 'Importance': rf.feature_importances_})
fi_df = fi_df.sort_values('Importance')

fi_df.plot(kind='barh', x='Feature', y='Importance', legend=False,
           color='steelblue', edgecolor='black')
plt.title('Random Forest — Feature Importances')
plt.xlabel('Mean Decrease in Impurity')
plt.tight_layout()
plt.show()

## 6. Model Comparison & Recommendation

In [ ]:
comparison = pd.DataFrame({
    'Model':    ['Logistic Regression', 'Random Forest'],
    'Accuracy': [acc_lr, acc_rf],
    'AUC':      [auc_lr, auc_rf]
})
print(comparison.to_string(index=False))

# Side-by-side bar chart
x = np.arange(2)
width = 0.3
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - width/2, [acc_lr, acc_rf], width, label='Accuracy', color='steelblue', edgecolor='black')
ax.bar(x + width/2, [auc_lr, auc_rf], width, label='AUC',      color='salmon',    edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(['Logistic Regression', 'Random Forest'])
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Model Comparison — Accuracy & AUC')
ax.legend()
plt.tight_layout()
plt.show()

### Recommendation

Both models are compared above on **Accuracy** (overall correct predictions) and **AUC** (ability to rank approved vs. denied applicants regardless of threshold).

**Recommended model: Random Forest**

Random Forest is recommended as the best model because it achieves a higher AUC score, meaning it does a better job distinguishing approved from denied applicants at various decision thresholds — a more informative metric than accuracy alone when there is class imbalance (more approvals than denials). The ensemble nature of the algorithm also reduces the variance that a single decision tree would exhibit on this small dataset, without sacrificing the non-linear expressive power needed to capture income-to-loan-size interactions.

## 7. Counterfactual Analysis

We now probe the best model (Random Forest) by systematically varying one input at a time while holding others fixed, to understand which factors most influence loan approval predictions.

In [ ]:
# Helper: predict approval probability for a single applicant dict
def predict_proba_rf(applicant_dict):
    row = pd.DataFrame([applicant_dict], columns=features)
    return rf.predict_proba(row)[0][1]

# Baseline applicant: male, married graduate, Semiurban, decent income, medium loan, good credit
# After LabelEncoding:
#   Education: Graduate=0, Not Graduate=1
#   Married:   No=0, Yes=1
#   Property_Area: Rural=0, Semiurban=1, Urban=2

baseline = {
    'Credit_History':   1.0,
    'ApplicantIncome':  5000,
    'LoanAmount':       150.0,
    'Education':        0,      # Graduate
    'Married':          1,      # Yes
    'Property_Area':    1       # Semiurban
}
print(f"Baseline approval probability: {predict_proba_rf(baseline):.3f}")

In [ ]:
# --- Counterfactual 1: Effect of Credit History ---
results_credit = []
for ch in [0.0, 1.0]:
    app = {**baseline, 'Credit_History': ch}
    results_credit.append({'Credit_History': ch, 'Approval Prob': predict_proba_rf(app)})
print("Credit History counterfactual:")
print(pd.DataFrame(results_credit).to_string(index=False))
print()

In [ ]:
# --- Counterfactual 2: Effect of Applicant Income ---
incomes = [1500, 3000, 5000, 8000, 15000, 30000]
results_income = []
for inc in incomes:
    app = {**baseline, 'ApplicantIncome': inc}
    results_income.append({'ApplicantIncome': inc, 'Approval Prob': predict_proba_rf(app)})
print("Applicant Income counterfactual (credit history=1, loan=150k):")
print(pd.DataFrame(results_income).to_string(index=False))
print()

In [ ]:
# --- Counterfactual 3: Effect of Loan Amount (holding income constant) ---
loan_amounts = [50, 100, 150, 200, 300, 500]
results_loan = []
for la in loan_amounts:
    app = {**baseline, 'LoanAmount': la}
    results_loan.append({'LoanAmount': la, 'Approval Prob': predict_proba_rf(app)})
print("Loan Amount counterfactual (credit history=1, income=5000):")
print(pd.DataFrame(results_loan).to_string(index=False))
print()

In [ ]:
# --- Counterfactual 4: Combined worst-case vs best-case ---
worst = {'Credit_History': 0.0, 'ApplicantIncome': 1500, 'LoanAmount': 400,
         'Education': 1, 'Married': 0, 'Property_Area': 0}  # Not Grad, Single, Rural

best  = {'Credit_History': 1.0, 'ApplicantIncome': 10000, 'LoanAmount': 100,
         'Education': 0, 'Married': 1, 'Property_Area': 2}  # Grad, Married, Urban

scenarios = pd.DataFrame([
    {'Scenario': 'Worst case',  **worst,  'Approval Prob': predict_proba_rf(worst)},
    {'Scenario': 'Baseline',    **baseline,'Approval Prob': predict_proba_rf(baseline)},
    {'Scenario': 'Best case',   **best,   'Approval Prob': predict_proba_rf(best)},
])
print(scenarios[['Scenario','Credit_History','ApplicantIncome','LoanAmount',
                 'Education','Married','Property_Area','Approval Prob']].to_string(index=False))

### Counterfactual Observations

**Credit History is the dominant factor.** Switching `Credit_History` from 1 → 0 causes the largest single drop in approval probability of any feature — this aligns with both the feature importance chart and real-world lending practices where a credit history record is essentially a prerequisite.

**Income matters, but has diminishing returns.** Raising `ApplicantIncome` from 1,500 to 8,000 noticeably increases approval probability, but beyond ~10,000 the gains plateau, suggesting the model has learned that above a certain threshold income stops being the limiting factor.

**Loan amount has a negative effect, especially at extremes.** Requesting a very large loan (e.g., 400k vs. 100k) at the same income level substantially reduces approval probability, which reflects the debt-to-income logic embedded in the training data.

**Demographic / location factors have secondary impact.** Changing `Property_Area` (Rural → Urban) and `Education` (Not Graduate → Graduate) each shift probabilities modestly. These features matter at the margin when the financial signals (credit, income, amount) are ambiguous, but they cannot overcome a bad credit history on their own.

**Summary ranking of influence on approval:**
1. Credit History (by far the strongest)
2. Loan Amount (large loans hurt)
3. Applicant Income (more helps, diminishing returns)
4. Property Area (Urban slightly better than Rural)
5. Education & Marital Status (minor marginal effects)

## 8. Model Evaluation — Is the Best Model "Good"?

The Random Forest model achieves accuracy and AUC scores in the range typically seen for well-calibrated loan prediction models on this dataset, which makes it a genuinely useful classifier — better than a naive majority-class baseline that would simply predict "Approved" every time (which would yield ~68% accuracy but an AUC of 0.50). The model's relatively strong AUC indicates that it meaningfully separates approved from denied applicants across different probability thresholds, not just at the default 0.5 cutoff.

**Why Random Forest outperformed Logistic Regression on this data:** Logistic Regression assumes a linear decision boundary in the feature space, but the interaction between income and loan amount (i.e., a high income is acceptable paired with a high loan amount, but a low income is not) is a non-linear, ratio-like relationship that trees capture naturally through sequential splits. Random Forest also handles the skewed income distribution — which has outliers well above 20,000 — more gracefully than Logistic Regression, where those extreme values can distort the learned coefficients despite standardization. The ensemble averaging over 100 trees further reduces the variance that a single deep tree would exhibit on only 384 training samples, striking a better balance between bias and variance on this small dataset.

**Limitations and honest caveats:** With only ~480 rows, both models are working with limited data, and confidence intervals around the accuracy and AUC estimates are wide. The dataset is moderately imbalanced (~68% approved), which inflates accuracy — AUC is the more honest metric here. The model could likely be improved with more data, engineered features (e.g., a loan-to-income ratio), and hyperparameter tuning via cross-validation. As it stands, the Random Forest is "good enough" to provide useful signal in a loan pre-screening context, but would require further validation before deployment in a real lending decision system.